<a href="https://colab.research.google.com/github/udplabs/okta-ai-poc/blob/main/colabs/cross_app_authz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Okta Cross-App Access Demo

This notebook demonstrates the complete **Identity Assertion Authorization Grant (ID-JAG)** flow for secure cross-application access using the Okta AI SDK.


### The Sequence / Flow

1. **Get Access Token:** Authenticate as a user to obtain an access token
1. **Exchange Access Token for ID-JAG Token:** Convert user's Access token to a JWT Assertion Grant Token
1. **Verify ID-JAG Token:** Validate the token's authenticity and claims. 
   - This step is *technically* optional but we recommend it.
1. **Exchange ID-JAG for new Access Token:** Get an access token for the target resource
1. **Verify Access Token:** Validate final access token before use
   - This step would _normally_ be done by the resource. 
2. **Call Resource:** _Not included in this workbook._

> Training intent: this notebook prioritizes conceptual clarity and runnable sample code for ID-JAG.

```mermaid
sequenceDiagram
    autonumber
    participant U as User
    participant C as Client App
    participant CAS as Okta Client Authz Server
    participant A as Agent Principal
    participant OAS as Okta Org Authz Server
    participant RAS as Okta Resource Authz Server
    participant R as Resource

	rect rgb(0, 128, 255)
		Note right of U: Step 1
			U->>C: Authenticate via browser
			C->>CAS: Authorization Code + Token request
			CAS-->>C: User Access Token
	end

	rect rgb(170, 0, 255)
		Note right of C: Step 2
			C-->>A: Request + User Access Token
			A->>OAS: Exchange User Access Token for ID-JAG
			OAS-->>A: ID-JAG Token
	end

	rect rgb(0, 156, 112)
		Note left of A: Step 3
		opt 
			A->>A: Verify ID-JAG Token
		end
	end

	rect rgb(90, 49, 0)
		Note right of A: Step 4
		A->>RAS: Exchange ID-JAG for Resource Access Token
		RAS-->>A: Resource Access Token
	end
    
	rect rgb(255, 0, 119)
		Note left of A: Step 5
		opt
			A->>A: Verify Resource Access Token
			Note right of A: This verification would normally be done by the resource.
		end
	end

	rect rgb(38, 73, 109)
		Note right of A: Step 6
		A-->>R: Request to Resource + Access Token
	end
```

---

### Prerequisites

Before running this notebook, you need:

1. **Okta Organization** with:
   - Custom authorization server configured
   - OAuth 2.0 application with Token Exchange enabled
   - ID-JAG support enabled

2. **Agent/Workload Principal** with:
   - Principal ID (agent identifier)
   - Private JWK (RSA key pair for JWT bearer assertion)

3. **User**:
   - Valid Okta User to authenticate and obtain an access token.

## Setup and Installation


> [!IMPORTANT]
**Training Notebook Note**
- This notebook intentionally includes some sample-code shortcuts for readability and step-by-step learning.
- For production: store secrets outside notebooks, enforce full token verification, and package helper logic into tested modules.
- Before publishing to GitHub, remove local-only utilities and any environment-specific scaffolding.

### 1. Install the Okta Python SDK

In [ ]:
# Install the Okta AI SDK from PyPI

%pip install --upgrade okta-client-python

### 2. Configuration Variables

Set _all_ your configuration variables and _run the cell._

In [ ]:
# @title { display-mode: "form", vertical-output: true }
# @markdown _Enter your configuration variables here and click the 'run' to validate._
# @markdown <br><br> These will be used throughout the notebook.

OKTA_DOMAIN = 'https://atko-ai.oktapreview.com' # @param {"type":"string","placeholder":"Enter your entire Okta domain (including https)"}
# @markdown <br>

# @markdown ---
# @markdown ##### Client (App) configuration
# @markdown ---
CLIENT_ID = '' # @param {"type":"string","placeholder":"Enter your client Id"}
REDIRECT_URI = 'http://localhost:8080/authorization-code/callback' # @param {"type":"string","placeholder":"Enter application redirect URI"}
CLIENT_AUTHZ_SERVER_ID = '' # @param {"type":"string","placeholder":"Enter your authorization server Id"}
CLIENT_SCOPES = ['openid', 'profile', 'email', 'mcp:read'] # @param {"type":"raw","placeholder": "Enter the scopes your client should request."}

# @markdown <em>If you are using client secret authentication, enter your client secret below.</em>
CLIENT_SECRET = '' # @param {"type":"string","placeholder":"Enter your client secret"}
# @markdown <br><em>If you are using private key authentication, enter your application's private JWK below.</em>
CLIENT_PRIVATE_JWK = {} # @param {"type":"raw","placeholder": "{}"}
# @markdown <br>

# @markdown ---
# @markdown ##### Principal/Agent Configuration
# @markdown ---
PRINCIPAL_ID = '' # @param {"type":"string","placeholder":"Enter your agent identifier"}
RESOURCE_URI = '' # @param {"type":"string","placeholder":"Enter the audience/resource URL configured on the agent."}
PRINCIPAL_SCOPES = ['mcp:read'] # @param {"type":"raw","placeholder": "Enter the scopes your agent should request."}

# @markdown <em>If you are using client secret authentication, enter your agent secret below.</em>
PRINCIPAL_SECRET = '' # @param {"type":"string","placeholder":"Enter your agent's client secret"}
# @markdown <br><em>If you are using private key authentication, enter your agent's private JWK below.</em>
PRINCIPAL_PRIVATE_JWK = {} # @param {"type":"raw","placeholder": "{}"}
# @markdown <br>

# @markdown ---
# @markdown ##### Resource Server Configuration
# @markdown ---

RESOURCE_AUTHZ_SERVER_ID = 'ausubr9dq7o80RBml1d7' # @param {"type":"string","placeholder":"Enter your authorization server Id"}
RESOURCE_SERVER_AUDIENCE = 'https://benefits.streamward.com' # @param {"type":"string","placeholder":"Enter your resource server audience"}

# @markdown <br>
SDK_DEBUG_ENABLED = True # @param {"type":"boolean"}

# ===== Uncomment only for local dev and comment out the next section =====
from pathlib import Path
import sys

cwd = Path.cwd()
repo_root = cwd if (cwd / "utils.py").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import utils
validate_config = utils.validate_config
globals().setdefault("Debugger", utils.Debugger)
# ==============

# Import utility functions for validation and other operations
# === Uncomment this section to load utils.py from GitHub if you are not running this notebook locally ===
# import requests
# from typing import Callable
# url = "https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/utils.py"

# response = requests.get(url)
# if response.status_code == 200:
#     # This executes the code inside utils.py
#     exec(response.text)
#     print("Successfully loaded utils.py from GitHub!")
# else:
#     raise Exception(f"Failed to load utils.py. Please contact a code owner -- this is not good!")

# validate_config: Callable[[dict, str], None] | None = None
# ========

# Perform validation of the configuration variables
# Function will always exist in utils.py, but we check for its existence to avoid errors if the file fails to load.
if validate_config:
    try:
        validate_config(locals(), "authz")
    except ValueError as e:
        print(e)
        raise SystemExit("❌ Configuration validation failed. Please fix the above errors and re-run the cell.")

print("✅ All configuration variables validated successfully!")
print(f"    Okta Domain: {OKTA_DOMAIN}")
print(f"    Client ID: {CLIENT_ID}")
print(f"    Client Authorization Server: {CLIENT_AUTHZ_SERVER_ID}")

print(f"    Principal ID: {PRINCIPAL_ID}")
print(f"    Resource URI: {RESOURCE_URI}")
print(f"    Resource Authorization Server: {RESOURCE_AUTHZ_SERVER_ID}")
print(f"    Resource Server Audience: {RESOURCE_SERVER_AUDIENCE}")

global RESOURCE_ISSUER
global CLIENT_ISSUER
RESOURCE_ISSUER = f"{OKTA_DOMAIN}/oauth2/{RESOURCE_AUTHZ_SERVER_ID}"
CLIENT_ISSUER = f"{OKTA_DOMAIN}/oauth2/{CLIENT_AUTHZ_SERVER_ID}"


### 3. Initialize the SDK for _user authentication_

In [ ]:
# Initialize Okta SDK
from okta_client.authfoundation import OAuth2Client, OAuth2ClientConfiguration, ClientSecretAuthorization, LocalKeyProvider
from okta_client.authfoundation.oauth2.jwt_bearer_claims import JWTBearerClaims
from okta_client.authfoundation.oauth2.client_authorization import ClientAssertionAuthorization
from okta_client.oauth2auth import AuthorizationCodeContext, AuthorizationCodeFlow, CrossAppAccessFlow, CrossAppAccessTarget, Prompt

print("✅ Imports successful!")

STATE = globals().setdefault("STATE", {})

def build_client_authorization(client_id: str, client_secret: str, private_jwk: dict, audience: str, label: str):
    """Build client auth using private key if available, otherwise client secret."""
    if private_jwk and isinstance(private_jwk, dict) and private_jwk.get("kid"):
        print(f"\n✅ Using private key auth for {label} (kid={private_jwk.get('kid')})")
        return ClientAssertionAuthorization(
            assertion_claims=JWTBearerClaims(
                issuer=client_id,
                subject=client_id,
                audience=audience,
                expires_in=300
            ),
            key_provider=LocalKeyProvider(
                key=private_jwk,
                algorithm=private_jwk.get("alg", "RS256"),
                key_id=private_jwk.get("kid")
            )
        )

    print(f"\n✅ Using client secret auth for {label}")
    return ClientSecretAuthorization(
        id=client_id,
        secret=client_secret
    )

client_authz = build_client_authorization(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    private_jwk=CLIENT_PRIVATE_JWK,
    audience=CLIENT_ISSUER,
    label="client application"
)

# Initialize the user SDK with the appropriate configuration, including the issuer for the custom authorization server.
user_sdk_config = OAuth2ClientConfiguration(
    issuer=CLIENT_ISSUER,
    scope=CLIENT_SCOPES,
    redirect_uri=REDIRECT_URI,
    client_authorization=client_authz
)

user_sdk = OAuth2Client(configuration=user_sdk_config)
STATE["user_sdk"] = user_sdk

print("✅ User SDK initialized!")

if SDK_DEBUG_ENABLED:
    debugger_cls = globals().get("Debugger")
    if debugger_cls is None:
        print("[WARN] Debugger not loaded; continuing without SDK listener.")
    else:
        user_sdk.listeners.add(debugger_cls()) # type: ignore

---

## Step 1: Obtain User Tokens

If you don't have user tokens yet, use this section to obtain them via OIDC.

### Important:
This step uses a **Custom Authorization Server**. 

This is the correct approach for obtaining the initial user token that will be used in the ID-JAG flow.

### The Flow:

1. **Build authorization URL** - Users authenticate using OIDC via the browser.
2. **Copy redirect URL from browser** - Copy the **entire** redirect URL after authentication.
3. **Exchange code for tokens** - Get ID token with issuer = Okta domain

```mermaid
sequenceDiagram
    autonumber
    participant U as User
    participant C as Client App
    participant CAS as Okta Client Authz Server

	rect rgb(0, 128, 255)
		Note right of U: Step 1
			U->>C: Authenticate via browser
			C->>CAS: Authorization Code + Token request
			CAS-->>C: User Access Token
	end
    
```

### ① Build Authorization URL and Authenticate via the browser

Run the following cell and then click the generated button to open the authorization URL in a new browser tab.

In [ ]:
from IPython.display import HTML, display # Ensure display is imported for the HTML button

async def authorize():

  try:
    # Build the authorization URL
    authorization_context = AuthorizationCodeContext(
        prompt=Prompt.LOGIN,
        # The `resource` parameter is used (per RFC8693) to specify the audience/resource for which the access token is requested. In this case, it is set to the RESOURCE_URI defined earlier.
        resource=RESOURCE_URI
    )

    global auth_flow
    auth_flow = AuthorizationCodeFlow(client=user_sdk)
    STATE["auth_flow"] = auth_flow

    authorization_url = await auth_flow.start(context=authorization_context)

    print("✅ Authorization URL generated!")
    print(f"\n{authorization_url}")
    print("\n" + "="*80)

    # Display clickable button
    html_button = f"""
    <div style="margin: 20px 0;">
        <a href="{authorization_url}" target="_blank" style="
            display: inline-block;
            padding: 15px 30px;
            background-color: #007bff;
            color: white;
            text-decoration: none;
            border-radius: 5px;
            font-weight: bold;
            font-size: 16px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.2);
        ">Click Here to Authenticate with Okta</a>
    </div>
    <div style="margin: 20px 0; padding: 15px; background-color: #fff3cd; border-left: 4px solid #ffc107; border-radius: 4px;">
        <strong>Instructions:</strong>
        <ol style="margin: 10px 0 0 0;">
            <li>Click the button above to open the authorization URL in a new tab</li>
            <li>Sign in with your Okta credentials</li>
            <li>After authentication, you'll be redirected to: <code>{REDIRECT_URI}</code></li>
            <li>Copy the <strong>code</strong> parameter from the URL (it will look like: <code>?code=ABC123...</code>)</li>
            <li>Paste the code in the next cell to exchange it for tokens</li>
        </ol>
    </div>
    """

    display(HTML(html_button))

    print("\nWhat to do next:")
    print("   1. Click the button above")
    print("   2. Sign in to Okta")
    print("   3. Copy the entire URL")
    print("   4. Paste it in the next cell")
    print("\nNote: The tokens will be issued by the Client Authorization Server")
    print(f"      Issuer: {CLIENT_ISSUER}")

  except Exception as e:

    print(f"❌ [ERROR]: {e}")
    raise

await authorize()

### ② Exchange Authorization Code for Tokens

After authenticating...
1. copy the entire URL
1. paste the URL below
1. and run this cell to obtain tokens

In [ ]:
# @title { display-mode: "form" }

REDIRECT_URL = '' # @param { type: "string", placeholder: "Insert entire URL string here."}

async def exchange_code_for_tokens():
  if not REDIRECT_URL:
    print("\n❌ No url provided! Please paste the entire URL containing the `code` and run this cell again.")
    return

  if "auth_flow" not in globals() and not STATE.get("auth_flow"):
    raise SystemExit("❌ Missing auth flow. Run Cell 14 (Build and Open Authorization URL) first.")

  print("\n" + "=" * 80)
  print("② Exchange Authorization Code for User Tokens")
  print("=" * 80)

  try:
      flow = STATE.get("auth_flow", auth_flow)
      token = await flow.resume(REDIRECT_URL)

      print("Token exchange successful!\n")
      print("Token Response:")
      print(f"   Token Type: {token.token_type}")
      print(f"   Expires In: {token.expires_in} seconds")
      print(f"   Scope: {token.scope}")

      # Extract tokens
      global ID_TOKEN, ACCESS_TOKEN, REFRESH_TOKEN
      ID_TOKEN = token.id_token.raw
      ACCESS_TOKEN = token.access_token
      REFRESH_TOKEN = token.refresh_token

      STATE["id_token"] = ID_TOKEN
      STATE["access_token"] = ACCESS_TOKEN
      STATE["refresh_token"] = REFRESH_TOKEN

      print(f"\nTokens Obtained:")
      if ID_TOKEN:
          print(f" ✅ ID Token: https://jwt.io#token={ID_TOKEN}")
      if ACCESS_TOKEN:
          print(f" ✅ Access Token: https://jwt.io#token={ACCESS_TOKEN}")
      if REFRESH_TOKEN:
          print(f" ✅ Refresh Token: {REFRESH_TOKEN[:50]}...")

      # Decode and display access token claims (optional)
      if ACCESS_TOKEN:
          import jwt
          decoded = jwt.decode(ACCESS_TOKEN, options={"verify_signature": False})
          print(f"\nAccess Token Claims:")
          print(f"   Subject: {decoded.get('sub')}")
          print(f"   Email: {decoded.get('email', 'N/A')}")
          print(f"   Name: {decoded.get('name', 'N/A')}")
          print(f"   Issuer: {decoded.get('iss')}")
          print(f"   Audience: {decoded.get('aud')}")

          # Verify issuer is the expected client authorization server issuer.
          if decoded.get('iss') == CLIENT_ISSUER:
              print(f"\n ✅ Token issued by Custom Authorization Server: {CLIENT_ISSUER}")
          else:
              print(f"\n ⚠️ Unexpected issuer: {decoded.get('iss')}")
              print(f"   Expected: {CLIENT_ISSUER}")

      print("\n" + "="*80)
      print("You can now use ACCESS_TOKEN in the cross-app access flow.")
      print("="*80)

      print("Please verify your configuration!")
      print("Cross-App Access Configuration")
      print("=" * 60)
      print(f"   Okta Domain: {OKTA_DOMAIN}")
      print(f"   Client ID: {CLIENT_ID}")
      print(f"   Client Auth Server: {CLIENT_AUTHZ_SERVER_ID}")

      print(f"\n   Principal ID: {PRINCIPAL_ID}")
      print(f"    Principal KID: {PRINCIPAL_PRIVATE_JWK.get('kid')}")
      print(f"    Resource URI: {RESOURCE_URI}")

      print(f"\n   Resource Auth Server: {RESOURCE_AUTHZ_SERVER_ID}")
      print(f"   Resource Server Audience: {RESOURCE_SERVER_AUDIENCE}")
      print("=" * 60)

  except Exception as e:
      print(f" ❌ Error during token exchange: {e}")
      print("\nTroubleshooting:")
      print("   • Make sure you copied the entire authorization code")
      print("   • Verify your redirect_uri matches what's registered in Okta")
      print("   • Check that the authorization code hasn't expired (valid for ~60 seconds)")
      print("   • Ensure your client_id and client_secret are correct")

await exchange_code_for_tokens()

---

## Step 2: Exchange Access Token for ID-JAG Token

The first step is to exchange a user's Okta access token for an ID-JAG token. This token represents the user's identity and can be used for cross-application access.

### What happens:
The SDK...
1. generates a JWT bearer assertion using your private key;
1. calls the resources **custom authorization server's** token endpoint;
1. exchanges access token for ID-JAG token with specified audience;
1. exchanges the ID-JAG for an access token;
1. validates the returned access token;
1. returns access token.

```mermaid
sequenceDiagram
    autonumber 4

	participant C as Client App
    participant A as Agent Principal
    participant OAS as Okta Org Authz Server

	rect rgb(170, 0, 255)
		Note right of C: Step 2
			C-->>A: Request + User Access Token
			A->>OAS: Exchange User Access Token for ID-JAG
			OAS-->>A: ID-JAG Token
	end

```

### Initialize the Okta _agent_ SDK

First, let's initialize a new instance of the SDK with cross-app access configuration for our *agent* specifically.

In [ ]:
if "build_client_authorization" not in globals():
    raise SystemExit("❌ Missing helper function. Re-run Cell 11 (SDK initialization) and then run this cell again.")

client_authz = build_client_authorization(
    client_id=PRINCIPAL_ID,
    client_secret=PRINCIPAL_SECRET,
    private_jwk=PRINCIPAL_PRIVATE_JWK,
    audience=f"{OKTA_DOMAIN}/oauth2/v1/token",
    label="agent principal"
)

# The SDK will use either private key JWT assertion or client secret auth to perform token exchange operations.
agent_sdk_config = OAuth2ClientConfiguration(
    issuer=OKTA_DOMAIN,
    client_authorization=client_authz,
    scope=PRINCIPAL_SCOPES
)

print("✅ OAuth2 client configuration created")

# Create OAuth2 client
agent_sdk = OAuth2Client(configuration=agent_sdk_config)
STATE["agent_sdk"] = agent_sdk

if SDK_DEBUG_ENABLED:
    debugger_cls = globals().get("Debugger")
    if debugger_cls is None:
        print("⚠️ WARN: Debugger not loaded; continuing without SDK listener.")
    else:
        agent_sdk.listeners.add(debugger_cls()) # type: ignore

print("✅ OAuth2 client created")
print("✅ Step 2.1 Complete: Agent SDK initialized successfully!")

### ❺ Exchange Token
Great! Now let's start the token exchange.

In [ ]:
print("\n" + "=" * 80)
print("❺ Exchange user's access token for an ID-JAG")
print("=" * 80)

if not STATE.get("access_token") and "ACCESS_TOKEN" not in globals():
    raise SystemExit("❌ Missing ACCESS_TOKEN. Run Step 1.2 first.")

if "agent_sdk" not in globals() and not STATE.get("agent_sdk"):
    raise SystemExit("❌ Missing agent SDK. Run Cell 20 (Initialize the Okta agent SDK) first.")

# Create target
agent_sdk_target = CrossAppAccessTarget(
    issuer=RESOURCE_ISSUER
)
STATE["agent_sdk_target"] = agent_sdk_target

print(f"✅ Target created: {agent_sdk_target.issuer}")

# Create cross-app flow
flow_client = STATE.get("agent_sdk", agent_sdk)
agent_sdk_flow = CrossAppAccessFlow(
    client=flow_client,
    target=agent_sdk_target
)
STATE["agent_sdk_flow"] = agent_sdk_flow

print("✅ Cross-app access flow created")

async def exchange_id_token():
  try:
    source_token = STATE.get("access_token", ACCESS_TOKEN)

    id_jag_result = await agent_sdk_flow.start(
       token=source_token,
       token_type="access_token",
       audience=RESOURCE_ISSUER,
       scope=PRINCIPAL_SCOPES
    )

    if id_jag_result.resume_assertion_claims is not None:
      print("⚠️ Flow requires manual assertion claims. This notebook expects automatic exchange.")
      print(f"   Required claims: {id_jag_result.resume_assertion_claims}")
      raise SystemExit("❌ Unable to continue automatically.")

    # Store for next step
    token_obj = agent_sdk_flow.context.id_jag_token
    if token_obj is None:
      raise SystemExit("❌ ID-JAG token was not returned. Check token exchange configuration and retry.")

    global id_jag_token
    id_jag_token = token_obj
    STATE["id_jag_token"] = id_jag_token

    print("\n✅ ID-JAG token obtained!")
    print(f"\nToken Details:")
    print(f"   Token Type: {id_jag_token.issued_token_type}")
    print(f"   Expires In: {id_jag_token.expires_in} seconds")
    print(f"   Scope: {id_jag_token.scope or 'N/A'}")
    print(f"   Decoded Token: https://jwt.io#token={id_jag_token.access_token}")

  except Exception as e:
      print(f"❌ [ERROR]: {e}")
      raise

await exchange_id_token()

---

## Step 3: Verify ID-JAG Token

Before using the ID-JAG token, we should verify its authenticity. This step:
- Validates the token signature using Okta's public keys (JWKS)
- Checks the audience, issuer, and expiration claims
- Extracts user information from the token

***SECURITY NOTE:*** *Always verify tokens before trusting their contents.* 

This prevents:
- Tampered tokens
- Expired tokens
- Tokens meant for different audiences

```mermaid
sequenceDiagram
    autonumber 7
    participant A as Agent Principal

	rect rgb(0, 156, 112)
		Note left of A: Step 3
		opt 
			A->>A: Verify ID-JAG Token
		end
	end

```

### Token Claims Reference

| Claim | Name | Meaning in This Notebook |
|---|---|---|
| `iss` | Issuer | Should match the expected Okta authorization server (i.e. org server) issuer for the current token. |
| `aud` | Audience | Must match the target resource or configured audience for this step. |
| `exp` | Expiration time | Token must still be valid. |
| `sub` | Subject | The entity the token references. The human identity in this flow. |
| `cid` | Client ID | Client/principal identifier (the workload or app identity). |
| `scp` | Scope | Granted scopes used for authorization decisions. |

In [ ]:
import time
import requests
import jwt
from datetime import datetime
from jwt import PyJWKClient

current_id_jag_token = STATE.get("id_jag_token") or globals().get("id_jag_token")
if current_id_jag_token is None:
    raise SystemExit("❌ Missing ID-JAG token. Run Step 2.2 first.")

print("\n" + "=" * 80)
print("⑦ Verify ID-JAG token")
print("=" * 80)

print(f"   Expected Audience: {agent_sdk_target.issuer}\n")

def decode_token_unverified(token: str) -> dict:
    """Decode JWT without verification for display only."""
    return jwt.decode(token, options={"verify_signature": False})

def discover_jwks_uri(expected_issuer: str) -> str:
    """Discover JWKS URI from the issuer metadata endpoint."""
    config_url = f"{expected_issuer}/.well-known/openid-configuration"
    response = requests.get(config_url, timeout=15)
    response.raise_for_status()
    jwks_uri = response.json().get("jwks_uri")
    if not jwks_uri:
        raise ValueError(f"No jwks_uri found in metadata for issuer: {expected_issuer}")
    return jwks_uri

def verify_jwt_with_jwks(token: str, expected_issuer: str, expected_audience: str, label: str) -> dict:
    """Verify JWT signature and registered claims using issuer JWKS."""
    jwks_uri = discover_jwks_uri(expected_issuer)
    jwk_client = PyJWKClient(jwks_uri)
    signing_key = jwk_client.get_signing_key_from_jwt(token)

    decoded = jwt.decode(
        token,
        signing_key.key,
        algorithms=["RS256", "RS384", "RS512", "ES256", "ES384", "ES512"],
        audience=expected_audience,
        issuer=expected_issuer,
    )

    print(f"✅ Signature verification passed for {label}")
    print(f"✅ Issuer matches: {expected_issuer}")
    print(f"✅ Audience matches: {expected_audience}")
    return decoded

decoded_jag_unverified = decode_token_unverified(current_id_jag_token.access_token)

# Keep educational claim visibility in output.
print("Decoded Claims Preview (unverified):")
print(f"   iss: {decoded_jag_unverified.get('iss')}")
print(f"   aud: {decoded_jag_unverified.get('aud')}")
print(f"   sub: {decoded_jag_unverified.get('sub')}")

try:
    decoded_jag = verify_jwt_with_jwks(
        token=current_id_jag_token.access_token,
        expected_issuer=RESOURCE_ISSUER,
        expected_audience=RESOURCE_ISSUER,
        label="ID-JAG token",
    )
except Exception as e:
    print(f"❌ Signature/claim verification failed: {e}")
    raise

# Check expiration for readable display.
exp = decoded_jag.get("exp")
if exp and exp > time.time():
    exp_time = datetime.fromtimestamp(exp)
    print(f"✅ Token valid until: {exp_time.strftime('%Y-%m-%d %H:%M:%S')}")
else:
    print("⚠️ Token expired!")

---
## Step 4: Exchange ID-JAG Token for Authorization Server Token

Now we exchange the verified ID-JAG token for an access token from a custom authorization server. This token can be used to access protected resources.

### What happens:
1. SDK generates a JWT bearer assertion for the authorization server (*or uses client_id/client_secret if configured*)
2. Uses the ID-JAG token as the assertion
3. Calls the custom authorization server's token endpoint
4. Returns an access token with appropriate scopes

```mermaid
sequenceDiagram
    autonumber 8
    participant A as Agent Principal
	participant RAS as Okta Resource Authz Server

	rect rgb(90, 49, 0)
		Note right of A: Step 4
		A->>RAS: Exchange ID-JAG for Resource Access Token
		RAS-->>A: Resource Access Token
	end
```

### Why this matters:
This is useful when you need to access resources protected by a custom authorization server with specific scopes and policies.

In [ ]:
async def exchange_id_jag_token():
  """Exchanges an ID-JAG token for resource access token."""
  try:

    if "agent_sdk_flow" not in globals() and not STATE.get("agent_sdk_flow"):
      raise SystemExit("❌ Missing cross-app flow. Run Step 2.2 first.")
    if "id_jag_token" not in globals() and not STATE.get("id_jag_token"):
      raise SystemExit("❌ Missing ID-JAG token. Run Step 2.2 first.")

    print("\n" + "=" * 80)
    print("⑧Exchange ID-JAG for resource access token")
    print("=" * 80)

    flow = STATE.get("agent_sdk_flow", agent_sdk_flow)
    auth_server_result = await flow.resume()

    print("✅ Authorization server token obtained!")
    print("Token Details:")
    print(f"   Token Type: {auth_server_result.token_type}")
    print(f"   Expires In: {auth_server_result.expires_in} seconds")
    print(f"   Scope: {auth_server_result.scope or 'N/A'}")
    print(f"   Decoded Token: https://jwt.io#token={auth_server_result.access_token}")

    print(f"   Refresh Token: {'✅' if auth_server_result.refresh_token else '❌'}")

    # Store for next step
    global auth_server_token
    auth_server_token = auth_server_result.access_token
    STATE["auth_server_result"] = auth_server_result
    STATE["auth_server_token"] = auth_server_token

  except Exception as e:
      print(f"[ERROR] Error: {e}")
      raise

await exchange_id_jag_token()

---
## Step 5: Verify Resource Access Token

The final step is to verify the authorization server token before using it to access resources.

### What happens:
1. Validates the token signature using the authorization server's public keys
2. Checks audience, issuer, and expiration
3. Extracts scope and user information

```mermaid
sequenceDiagram
    autonumber 10
    
    participant A as Agent Principal
    
	rect rgb(255, 0, 119)
		Note left of A: Step 5
		opt
			A->>A: Verify Resource Access Token
			Note right of A: This verification would normally be done by the resource.
		end
	end

```

### Important:
The resource server (*your API*) should perform this verification on every request to ensure:
- Token is valid and not tampered with
- Token is not expired
- Token is intended for this resource server (*audience check*)
- User has required permissions (*scope check*)

In [ ]:
print("\n" + "=" * 80)
print("①⓪ Verify Access Token")
print("=" * 80)

if "verify_jwt_with_jwks" not in globals():
    raise SystemExit("❌ Missing verification helper. Run Step 3 first.")

if "auth_server_token" not in globals() and not STATE.get("auth_server_token"):
    raise SystemExit("❌ Missing authorization server token. Run Step 4 first.")

token_to_verify = STATE.get("auth_server_token", auth_server_token)

decoded_access_unverified = decode_token_unverified(token_to_verify)
print("Decoded Claims Preview (unverified):")
print(f"   iss: {decoded_access_unverified.get('iss')}")
print(f"   aud: {decoded_access_unverified.get('aud')}")
print(f"   sub: {decoded_access_unverified.get('sub')}")

decoded_access = verify_jwt_with_jwks(
    token=token_to_verify,
    expected_issuer=RESOURCE_ISSUER,
    expected_audience=RESOURCE_SERVER_AUDIENCE,
    label="resource access token",
)

# Check expiration
exp = decoded_access.get('exp')
if exp and exp > time.time():
    exp_time = datetime.fromtimestamp(exp)
    print(f"✅ Token valid until: {exp_time.strftime('%Y-%m-%d %H:%M:%S')}")
else:
    print("⚠️  Token expired or no expiration!")

# Check scopes
scope_claim = decoded_access.get('scp', decoded_access.get('scope', ''))
if isinstance(scope_claim, list):
    scopes = scope_claim
elif isinstance(scope_claim, str):
    scopes = scope_claim.split()
else:
    scopes = []

if scopes:
    print(f"\n✅ Granted Scopes: {', '.join(scopes)}")
    if 'mcp:read' in scopes:
        print("✅ Has 'mcp:read' permission")

print(decoded_access.get('cid'))
print(decoded_access.get('sub'))

**Note** 
The ID JAG is a short lived one time use token. Access token contains the sub ("human in the loop") and cid ("AI agent workload principal").  Review syslog for audit log.  

##### Congratulations on successful completion of the cross app access flow and securing your AI Agents with Okta!

---

## Resources

- [Okta AI SDK Documentation](https://github.com/okta/okta-client-python/tree/main)
- [ID-JAG - Identity Assertion Authorization Grant](https://datatracker.ietf.org/doc/draft-ietf-oauth-identity-assertion-authz-grant/)

---
